# Training a Basic Classifier on Clidemia Hirta Leaves
This notebook should give us a classification baseline on our dataset 

In [4]:
import os
import tempfile

# Set tempfile
tempfile.tempdir = "/local/scratch/carlyn.1/tmp"

# Setting GPU
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

## Training Loop

In [5]:
from dataclasses import dataclass
from pathlib import Path


import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import SGD
from torch.nn import BCELoss
import torchvision.transforms as T
import pandas as pd

from inv_plts.data.datasets import BasicInvasivePlantsDataset
from inv_plts.models.simple import BasicInvasiveSpeciesCNN
from inv_plts.utils import create_data_splits_from_df

print("Loading Model")
# net = BasicInvasiveSpeciesCNN().cuda()


class BasicInvasiveSpeciesCNNv2(nn.Module):
    def __init__(
        self,
        in_dims=3,
        layer_feature_dims=[16, 32, 64, 128, 256],
        layer_widths=[3, 3, 3, 3, 3],
    ):
        super().__init__()
        self.sigmoid = nn.Sigmoid()
        self.maxpool = nn.MaxPool2d(kernel_size=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.layers = []
        for i, (feat_dims, width) in enumerate(zip(layer_feature_dims, layer_widths)):
            if i == 0:
                in_dim = in_dims
            else:
                in_dim = layer_feature_dims[i - 1]

            self.layers.append(
                self._construct_layer(
                    in_dim=in_dim, out_dim=feat_dims, layer_width=width
                )
            )

        self.layer = nn.ModuleList(self.layers)
        self.fc = nn.Linear(in_features=256, out_features=5, bias=True)

    def _construct_layer(self, in_dim, out_dim, layer_width):
        dims = [(in_dim, out_dim)] + [
            (out_dim, out_dim) for _ in range(layer_width - 1)
        ]
        inner_layers = [
            nn.Sequential(
                nn.Conv2d(in_d, out_d, kernel_size=3), nn.BatchNorm2d(out_d), nn.ReLU()
            )
            for in_d, out_d in dims
        ]

        return nn.Sequential(*inner_layers, nn.MaxPool2d(kernel_size=2))

    def forward(self, x):
        h = x
        for layer in self.layers:
            h = layer(h)

        feats = torch.flatten(self.avgpool(h), start_dim=1)
        out = self.sigmoid(self.fc(feats))
        return out


net = BasicInvasiveSpeciesCNNv2().cuda()

image_root_path = Path("/local/scratch/carlyn.1/invasive-species-leaves")
# image_root_path = Path("img_tmp")
metadata_csv = Path("../tmp/metadata/linked_metadata.csv")

transforms = T.Compose(
    [
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

print("Loading Dataset")
df = pd.read_csv(metadata_csv)
train_df, val_df, test_df = create_data_splits_from_df(df)

train_dataset = BasicInvasivePlantsDataset(
    image_root=image_root_path, df=train_df, transform=transforms
)
val_dataset = BasicInvasivePlantsDataset(
    image_root=image_root_path, df=val_df, transform=transforms
)
test_dataset = BasicInvasivePlantsDataset(
    image_root=image_root_path, df=test_df, transform=transforms
)
optimizer = SGD(net.parameters(), lr=0.003)
loss_fn = BCELoss()


@dataclass
class BatchStructure:
    images: torch.Tensor
    labels: torch.Tensor


def collate_fn(batch):
    images = []
    labels = []
    for item in batch:
        images.append(item.image)
        labels.append(item.label)

    return BatchStructure(images=torch.stack(images), labels=torch.stack(labels))


train_dataloader = DataLoader(
    train_dataset, batch_size=8, num_workers=4, shuffle=True, collate_fn=collate_fn
)
val_dataloader = DataLoader(
    val_dataset, batch_size=8, num_workers=4, shuffle=False, collate_fn=collate_fn
)
test_dataloader = DataLoader(
    test_dataset, batch_size=8, num_workers=4, shuffle=False, collate_fn=collate_fn
)

Loading Model
Loading Dataset


/home/carlyn.1/code/invasives-project/src/inv_plts/data/datasets.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df = self.df.fillna("NA")


In [6]:
from typing import Any, List
from dataclasses import dataclass, field


@dataclass
class TrainingTracker:
    total: int = 0
    total_loss: int = 0
    total_correct: List[Any] = field(default_factory=lambda: [])

In [7]:
# dataset.preprocess("img_tmp")

In [8]:
import plotly.graph_objects as go
import numpy as np

losses = []
accuracies = []
val_losses = []
val_accuracies = []


class LossGraph:
    def __init__(self):
        self.fig = go.FigureWidget(
            [
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines+markers",
                    name="Prediction Training Loss",
                ),
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines+markers",
                    name="Prediction Validation Loss",
                ),
            ]
        )

    def update(self):
        self.fig.data[0].x = list(range(len(losses)))
        self.fig.data[0].y = losses
        self.fig.data[1].x = list(range(len(val_losses)))
        self.fig.data[1].y = val_losses

    def display(self):
        display(self.fig)


class AccuracyGraph:
    def __init__(self):
        class_names = ["Healthly", "Leaf Miner", "Rust", "Other Insect", "Mechanical"]
        self.fig = go.FigureWidget(
            [
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines+markers",
                    name=f"Accuracy {class_names[i]}",
                )
                for i in range(5)
            ]
            + [
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines+markers",
                    name=f"Validation Accuracy {class_names[i]}",
                )
                for i in range(5)
            ]
        )

    def update(self):
        for cls_idx in range(5):
            self.fig.data[cls_idx].x = list(range(len(accuracies)))
            self.fig.data[cls_idx].y = np.array(accuracies)[:, cls_idx]
            self.fig.data[cls_idx + 5].x = list(range(len(val_accuracies)))
            self.fig.data[cls_idx + 5].y = np.array(val_accuracies)[:, cls_idx]

    def display(self):
        display(self.fig)


loss_graph = LossGraph()
loss_graph.display()

accuracy_graph = AccuracyGraph()
accuracy_graph.display()

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Prediction Training Loss',
              'type': 'scatter',
              'uid': '99269c45-0e43-49e6-ae6e-edad17e30558',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Prediction Validation Loss',
              'type': 'scatter',
              'uid': '8559e793-4ca3-4023-8b78-97f3e3145e69',
              'x': [],
              'y': []}],
    'layout': {'template': '...'}
})

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Accuracy Healthly',
              'type': 'scatter',
              'uid': 'a985efd8-33fd-4f76-b173-5efd5522e253',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Accuracy Leaf Miner',
              'type': 'scatter',
              'uid': 'a289189b-a9ab-4904-a49d-3f0a1c988abe',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Accuracy Rust',
              'type': 'scatter',
              'uid': '07ee42ce-b9f0-4e2d-84c9-95511708dd12',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Accuracy Other Insect',
              'type': 'scatter',
              'uid': '0e03e980-71fb-47e1-ab16-ff849b3e98e5',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Accuracy Mechanical',
       

In [ ]:
from tqdm import tqdm


losses = []
accuracies = []
val_losses = []
val_accuracies = []
NUM_EPOCHS = 200
tbar = tqdm(range(NUM_EPOCHS), desc="Epoch Training", position=1, leave=True)
for epoch in tbar:
    train_tracker = TrainingTracker()
    net.train()
    for data in tqdm(train_dataloader, desc="Batch Training", position=0, leave=False):
        # Training
        optimizer.zero_grad()
        out = net(data.images.cuda())
        loss = loss_fn(out, data.labels.cuda())
        loss.backward()
        optimizer.step()

        damage_predicted = (out >= 0.5).detach().cpu().type(torch.LongTensor)
        correct = damage_predicted == data.labels.detach().cpu().type(torch.LongTensor)

        # Tracking
        train_tracker.total += len(data.images)
        train_tracker.total_loss += loss.item()
        train_tracker.total_correct.append(correct.sum(0).numpy())

    net.eval()
    val_tracker = TrainingTracker()
    with torch.no_grad():
        for data in tqdm(
            val_dataloader, desc="Batch Validation", position=0, leave=False
        ):
            out = net(data.images.cuda())
            loss = loss_fn(out, data.labels.cuda())

            damage_predicted = (out >= 0.5).detach().cpu().type(torch.LongTensor)
            correct = damage_predicted == data.labels.detach().cpu().type(
                torch.LongTensor
            )

            # Tracking
            val_tracker.total += len(data.images)
            val_tracker.total_loss += loss.item()
            val_tracker.total_correct.append(correct.sum(0).numpy())

    # Tracking
    losses.append(train_tracker.total_loss)
    accuracy = np.stack(train_tracker.total_correct).sum(0) / train_tracker.total
    accuracies.append(accuracy)

    val_losses.append(val_tracker.total_loss)
    val_accuracy = np.stack(val_tracker.total_correct).sum(0) / val_tracker.total
    val_accuracies.append(val_accuracy)

    tbar.set_postfix(
        {
            "training-loss": train_tracker.total_loss,
            "validation-loss": val_tracker.total_loss,
        }
    )

    loss_graph.update()
    accuracy_graph.update()

net.eval()
test_tracker = TrainingTracker()
all_predictions = []
all_labels = []
with torch.no_grad():
    for data in tqdm(test_dataloader, desc="Batch Testing", position=0, leave=False):
        out = net(data.images.cuda())
        loss = loss_fn(out, data.labels.cuda())

        damage_predicted = (out >= 0.5).detach().cpu().type(torch.LongTensor)
        correct = damage_predicted == data.labels.detach().cpu().type(torch.LongTensor)

        # Tracking
        test_tracker.total += len(data.images)
        test_tracker.total_loss += loss.item()
        test_tracker.total_correct.append(correct.sum(0).numpy())

        all_predictions.append(out.detach().cpu().numpy())
        all_labels.append(data.labels.detach().cpu().numpy())

all_predictions = np.concatenate(all_predictions)
all_labels = np.concatenate(all_labels)

testing_accuracies = np.stack(test_tracker.total_correct).sum(0) / test_tracker.total
print(
    f"Testing Loss: {test_tracker.total_loss} | Testing Accuracy: {np.array(test_tracker.total_correct) / test_tracker.total}"
)

save_dir = Path("/home/carlyn.1/code/invasives-project/tmp")
torch.save(net.state_dict(), save_dir / "model.pt")

Batch Training:  32%|███▏      | 37/117 [00:00<00:01, 53.37it/s] 

In [ ]:
save_dir = Path("/home/carlyn.1/code/invasives-project/tmp")
net.load_state_dict(torch.load(save_dir / "model.pt"))
net.eval()
net = net.cuda()

In [ ]:
from tqdm import tqdm
import numpy as np

net.eval()
test_tracker = TrainingTracker()
all_predictions = []
all_labels = []
with torch.no_grad():
    for data in tqdm(test_dataloader, desc="Batch Testing", position=0, leave=False):
        out = net(data.images.cuda())
        loss = loss_fn(out, data.labels.cuda())

        damage_predicted = (out >= 0.5).detach().cpu().type(torch.LongTensor)
        correct = damage_predicted == data.labels.detach().cpu().type(torch.LongTensor)

        # Tracking
        test_tracker.total += len(data.images)
        test_tracker.total_loss += loss.item()
        test_tracker.total_correct.append(correct.sum(0).numpy())

        all_predictions.append(out.detach().cpu().numpy())
        all_labels.append(data.labels.detach().cpu().numpy())

all_predictions = np.concatenate(all_predictions)
all_labels = np.concatenate(all_labels)

testing_accuracies = np.stack(test_tracker.total_correct).sum(0) / test_tracker.total
print(
    f"Testing Loss: {test_tracker.total_loss} | Testing Accuracy: {np.array(test_tracker.total_correct) / test_tracker.total}"
)

Testing Loss: 8.292744954524096 | Testing Accuracy: [[0.02985075 0.02985075 0.02985075 0.0261194  0.02985075]
 [0.02238806 0.02985075 0.0261194  0.02985075 0.02985075]
 [0.0261194  0.0261194  0.02238806 0.0261194  0.02238806]
 [0.02985075 0.02985075 0.02985075 0.02985075 0.02985075]
 [0.02985075 0.0261194  0.0261194  0.0261194  0.02985075]
 [0.02985075 0.02985075 0.02985075 0.02985075 0.02985075]
 [0.02985075 0.02985075 0.0261194  0.02238806 0.0261194 ]
 [0.02985075 0.0261194  0.02985075 0.02238806 0.02985075]
 [0.02985075 0.02985075 0.02238806 0.02985075 0.02985075]
 [0.02985075 0.0261194  0.02985075 0.0261194  0.02985075]
 [0.02985075 0.0261194  0.02238806 0.0261194  0.02985075]
 [0.02985075 0.02985075 0.02985075 0.02985075 0.02985075]
 [0.02985075 0.0261194  0.02985075 0.0261194  0.0261194 ]
 [0.02985075 0.0261194  0.02985075 0.0261194  0.02985075]
 [0.0261194  0.02985075 0.02238806 0.02985075 0.02238806]
 [0.0261194  0.0261194  0.02985075 0.02985075 0.0261194 ]
 [0.02985075 0.02238

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
)


def calculate_metrics(y_true, y_pred, y_score):
    # Calculate precision, recall, F1 score, and accuracy
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    accuracy = accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_score)

    return precision, recall, f1, accuracy, roc_auc


label_names = test_dataset.get_labels()
for i, label in enumerate(label_names):
    y_true = all_labels[:, i]
    y_pred = all_predictions[:, i] >= 0.5
    y_score = all_predictions[:, i]

    precision, recall, f1, accuracy, roc_auc = calculate_metrics(
        y_true, y_pred, y_score
    )

    print(
        f"Label: {label} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1 Score: {f1:.4f} | Accuracy: {accuracy:.4f} | ROC AUC: {roc_auc:.4f}"
    )

Label: Healthy | Precision: 0.9291 | Recall: 0.8888 | F1 Score: 0.9073 | Accuracy: 0.9515 | ROC AUC: 0.9806
Label: Leaf Miner | Precision: 0.9154 | Recall: 0.9240 | F1 Score: 0.9196 | Accuracy: 0.9403 | ROC AUC: 0.9828
Label: Rust | Precision: 0.8557 | Recall: 0.8589 | F1 Score: 0.8569 | Accuracy: 0.8582 | ROC AUC: 0.9589
Label: Other Insect | Precision: 0.8893 | Recall: 0.8695 | F1 Score: 0.8784 | Accuracy: 0.8993 | ROC AUC: 0.9704
Label: Mechanical | Precision: 0.9038 | Recall: 0.8398 | F1 Score: 0.8672 | Accuracy: 0.9291 | ROC AUC: 0.9713


# Baseline
Testing Loss: 8.04

Accuracies:
- Healthy: 95.24%
- Leaf Miner: 71.4%
- Rust: 68.6%
- Other Insect: 69.5%
- Mechanical Damage: 84.8%